# Embedding Similarity Metrics — Cosine vs Dot Product vs Euclidean

A **clean, minimal** notebook to teach the three ways of comparing embedding vectors in a RAG pipeline.

We reuse the exact setup from `live_rag.ipynb`:
- Same PDF: *Gopalswamy Doraiswamy Naidu - Wikipedia.pdf*
- Same embedding model: `nomic-embed-text` (via Ollama)
- Same chunking: `RecursiveCharacterTextSplitter`

In `live_rag.ipynb` we only computed **cosine similarity** by hand. Here we write the
formula for the other two metrics as well, and compare the scores side by side.

| Property | Cosine | Dot Product | Euclidean |
|---|---|---|---|
| Measures | Angle only | Angle + magnitude | Absolute distance |
| Magnitude sensitive | No | Yes | Yes |
| Range | -1 to 1 | -inf to +inf | 0 to +inf |
| Similarity or Distance | Similarity (higher = closer) | Similarity (higher = closer) | Distance (lower = closer) |
| Normalized vectors | = Dot product | = Cosine | Still different |
| RAG use case fit | Best | Good (if normalized) | Okay |

## 1. Load the embedding model

We only need the embedding model for this lesson — no LLM required.

In [1]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")
embeddings

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OllamaEmbeddings(model='nomic-embed-text', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

## 2. Load and chunk the PDF

Exactly as in `live_rag.ipynb`: load the PDF, then split it into ~1000-character chunks.

In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = "A._P._J._Abdul_Kalam.pdf"

documents = PyPDFLoader(PDF_PATH).load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(documents)

print(f"Loaded {len(documents)} pages -> split into {len(chunks)} chunks")

Loaded 30 pages -> split into 136 chunks


## 3. Embed the query and the chunks

An embedding turns text into a list of numbers (a vector). Similar meaning -> similar vectors.
We embed the question once, and every chunk once.

In [11]:
query = "What are the Awards won by APJ Abdul Kalam"

query_embedding = embeddings.embed_query(query)
chunk_embeddings = [embeddings.embed_query(c.page_content) for c in chunks]

print(f"Each vector has {len(query_embedding)} dimensions")
print(f"We embedded 1 query + {len(chunk_embeddings)} chunks")

Each vector has 768 dimensions
We embedded 1 query + 136 chunks


## 4. The three metrics — written from scratch

No libraries, just the raw math so students can see exactly what each one does.
Let `a` = query vector, `b` = chunk vector.

### Cosine similarity (the one used in `live_rag.ipynb`)
Measures the **angle** between vectors. Ignores length.

$$\text{cosine}(a,b) = \frac{\sum a_i b_i}{\sqrt{\sum a_i^2}\;\sqrt{\sum b_i^2}}$$

### Dot product
Just the numerator of cosine. Rewards both alignment **and** magnitude.

$$\text{dot}(a,b) = \sum a_i b_i$$

### Euclidean distance
The straight-line distance between the two points. This is a **distance**, so
**smaller = more similar** (the opposite direction to the other two).

$$\text{euclidean}(a,b) = \sqrt{\sum (a_i - b_i)^2}$$

In [12]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot / (norm_a * norm_b)

In [13]:
def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

In [14]:
def euclidean_distance(a, b):
    return sum((x - y) ** 2 for x, y in zip(a, b)) ** 0.5

## 5. Score every chunk with all three metrics

For each chunk we compute all three numbers against the same query.

In [15]:
scores = []
for i, emb in enumerate(chunk_embeddings):
    scores.append({
        "chunk": i,
        "cosine": cosine_similarity(query_embedding, emb),
        "dot": dot_product(query_embedding, emb),
        "euclidean": euclidean_distance(query_embedding, emb),
    })

print(f"{"chunk":>5} | {"cosine":>10} | {"dot":>10} | {"euclidean":>10}")
print("-" * 48)
for s in scores:
    print(f'{s["chunk"]:>5} | {s["cosine"]:>10.4f} | {s["dot"]:>10.4f} | {s["euclidean"]:>10.4f}')

chunk |     cosine |        dot |  euclidean
------------------------------------------------
    0 |     0.5308 |     0.5308 |     0.9687
    1 |     0.5669 |     0.5669 |     0.9307
    2 |     0.5916 |     0.5916 |     0.9037
    3 |     0.4667 |     0.4667 |     1.0328
    4 |     0.5142 |     0.5142 |     0.9857
    5 |     0.5731 |     0.5731 |     0.9240
    6 |     0.4848 |     0.4848 |     1.0151
    7 |     0.4871 |     0.4871 |     1.0129
    8 |     0.4216 |     0.4216 |     1.0755
    9 |     0.4585 |     0.4585 |     1.0407
   10 |     0.5283 |     0.5283 |     0.9713
   11 |     0.4803 |     0.4803 |     1.0195
   12 |     0.5130 |     0.5130 |     0.9869
   13 |     0.5302 |     0.5302 |     0.9694
   14 |     0.5845 |     0.5845 |     0.9115
   15 |     0.6266 |     0.6266 |     0.8642
   16 |     0.5840 |     0.5840 |     0.9122
   17 |     0.5671 |     0.5671 |     0.9305
   18 |     0.6093 |     0.6093 |     0.8840
   19 |     0.4858 |     0.4858 |     1.0141
   20 

## 6. Which chunk does each metric pick?

Remember the direction of "best":
- **Cosine** and **Dot product** -> pick the **highest** score.
- **Euclidean** -> pick the **lowest** score (it's a distance).

We print the top-3 for each so students can see whether the metrics agree.

In [16]:
def show_top(title, key, reverse):
    ranked = sorted(scores, key=lambda s: s[key], reverse=reverse)
    print(f"\nTop 3 by {title}:")
    for s in ranked[:3]:
        preview = chunks[s['chunk']].page_content[:70].replace('\n', ' ')
        print(f"  chunk {s['chunk']:>2}  ({key}={s[key]:.4f})  {preview}...")

show_top("Cosine similarity (higher = better)", "cosine", reverse=True)
show_top("Dot product (higher = better)", "dot", reverse=True)
show_top("Euclidean distance (lower = better)", "euclidean", reverse=False)


Top 3 by Cosine similarity (higher = better):
  chunk 15  (cosine=0.6266)  Kalam with prime minister designate Manmohan Singh in New Delhi on 19 ...
  chunk 18  (cosine=0.6093)  posts supporting his candidature.[72][73] While the ruling Indian Nati...
  chunk 33  (cosine=0.6082)  A. P. J. Abdul Kalam; Poonam Kohli (2012). You are Unique: Scale New H...

Top 3 by Dot product (higher = better):
  chunk 15  (dot=0.6266)  Kalam with prime minister designate Manmohan Singh in New Delhi on 19 ...
  chunk 18  (dot=0.6093)  posts supporting his candidature.[72][73] While the ruling Indian Nati...
  chunk 33  (dot=0.6082)  A. P. J. Abdul Kalam; Poonam Kohli (2012). You are Unique: Scale New H...

Top 3 by Euclidean distance (lower = better):
  chunk 15  (euclidean=0.8642)  Kalam with prime minister designate Manmohan Singh in New Delhi on 19 ...
  chunk 18  (euclidean=0.8840)  posts supporting his candidature.[72][73] While the ruling Indian Nati...
  chunk 33  (euclidean=0.8852)  A. P. J. Abd

## 7. Takeaways for RAG

1. **Cosine** is the safe default — it compares *meaning* (direction) and ignores how long the text is.
2. **Dot product** is what vector databases actually run under the hood; it equals cosine *when vectors are normalized*, and it's faster.
3. **Euclidean** works too, but it's a *distance* — remember to sort ascending, and it's sensitive to vector magnitude.

For most embedding models (including `nomic-embed-text`), cosine similarity is the recommended metric.